# 04 — MCP Tools (stdio transport)

Shows how to:
1. Connect to an **MCP server** launched as a subprocess (stdio transport)
2. Auto-discover all tools the server exposes
3. Generate an LLM response that uses those tools

This example uses the public `@modelcontextprotocol/server-filesystem` package.

**Prerequisites**: `node` / `npx` available on `PATH` and `OPENAI_API_KEY` set.

In [1]:

from ravi.integrations.mcp import MCPClient, MCPTool
from ravi.integrations.llm.factory import create_model_client
from ravi.core.messages.client_messages import UserMessage, SystemMessage
from ravi.core.messages.content import TextBlock

from ravi.configs.settings import settings

CHAT_MODEL = settings.CHAT_MODEL
API_KEYS = {
    "openai":     settings.OPENAI_API_KEY,
    "anthropic":  settings.ANTHROPIC_API_KEY,
    "google":     settings.GEMINI_API_KEY,
    "groq":       settings.GROQ_API_KEY,
    "openrouter": settings.OPENROUTER_API_KEY,
}

ValidationError: 1 validation error for Settings
JWT_SECRET
  Value error, JWT_SECRET must be set to a strong random secret (min 32 chars). Generate one with: openssl rand -hex 32 [type=value_error, input_value='CHANGE_ME_IN_PRODUCTION_..._A_STRONG_RANDOM_SECRET', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

## Connect and discover tools

In [ ]:
async def demo():
    mcp_client = MCPClient()
    try:
        await mcp_client.connect(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
        )
        print("Connected to MCP server")

        mcp_tools = await mcp_client.discover_tools()
        print(f"Discovered {len(mcp_tools)} tools:")
        for t in mcp_tools:
            print(f"  - {t.name}: {t.description}")

        client = create_model_client(CHAT_MODEL, api_keys=API_KEYS)
        messages = [
            SystemMessage(content="You are a helpful assistant with filesystem access."),
            UserMessage(content=[TextBlock(text="List the files in the /tmp directory")]),
        ]

        response = await client.generate(
            messages=messages,
            tools=[t.get_openai_schema() for t in mcp_tools],
        )
        print(f"\nConfigured chat model: {CHAT_MODEL}")
        print(f"Agent response: {response.content}")
        if response.tool_calls:
            print("Tool calls requested:")
            for tc in response.tool_calls:
                print(f"  - {tc.name}({tc.arguments})")

    except Exception as e:
        print(f"Error: {e}")
        print("  Requires: Node.js + npx")
    finally:
        if mcp_client.is_connected:
            await mcp_client.disconnect()
            print("Disconnected")

await demo()

⚠ Error: Failed to connect to MCP server via stdio: fileno
  Requires: Node.js + npx  (npm install -g npx)


---
## API update notes (Sprint 5)

| Old API | New API | Notes |
|---|---|---|
| `MCPTool.from_mcp_client(client)` | `client.discover_tools()` | Returns same `list[MCPTool]`; prefer the new form |
| `ravi.extensions.mcp` | `ravi.integrations.mcp` | Import path changed |
| `ravi.providers.llm.openai` | `ravi.integrations.llm.openai` | Import path changed |


In [ ]:
# New: register MCP tools directly into AgentCatalog
from ravi.core.agent_catalog import AgentCatalog
from ravi.integrations.mcp.adapter import MCPCatalogAdapter

catalog = AgentCatalog()
adapter = MCPCatalogAdapter(catalog, namespace="filesystem")
print("MCPCatalogAdapter ready — call await adapter.register(mcp_client) after connecting")
print("All registered tools visible via catalog.all_tools()")
